# Part 1: Know the workload

Back-of-the-envelope estimates for one GPU on `traces/azure_conversation.csv` (Llama-3-8B, one A100-80GB). Fill the four TODOs in `analysis/workload.py` first (`pytest tests/test_workload.py`); this notebook only calls them.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis import workload as wl
from analysis.workload import arrival_rates, kv_bytes_per_token, kv_in_use, load_trace, predict_queue_wait

df = load_trace(ROOT / "traces" / "azure_conversation.csv")
span = df.arrived_at.max() - df.arrived_at.min()
print(f"n={len(df)}  span={span:.0f}s  native QPS={(len(df) - 1) / span:.2f}")

## Arrivals over time

In [ ]:
BIN_S = 30.0
rates = arrival_rates(df, BIN_S)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
ax0.bar(rates.bin_start, rates.req_per_s, width=BIN_S, align="edge", color="#555555")
ax0.set_ylabel("requests / s")
ax1.plot(rates.bin_start, rates.prompt_tok_per_s, label="prompt tok/s")
ax1.plot(rates.bin_start, rates.output_tok_per_s, label="output tok/s")
ax1.set_ylabel("tokens / s")
ax1.set_xlabel("trace time (s)")
ax1.legend(frameon=False)
fig.tight_layout()
plt.show()

## Predicted queue wait (one GPU)

In [ ]:
WAIT_TARGET_S = 2.0   # P99 wait target used for the rest of the assignment

w = predict_queue_wait(df)
print(f"predicted wait P50 {np.median(w):.2f}s  P99 {np.quantile(w, 0.99):.2f}s  max {w.max():.1f}s")

def longest_late_stretch(t, wait, merge_gap_s=60.0):
    """Longest time range where requests wait over the target (gaps under a minute merged).
    Part 2's notebook uses the same rule."""
    late = t[wait > WAIT_TARGET_S]
    if len(late) == 0:
        return "never"
    breaks = np.flatnonzero(np.diff(late) > merge_gap_s)
    starts, ends = np.r_[late[0], late[breaks + 1]], np.r_[late[breaks], late[-1]]
    i = np.argmax(ends - starts)
    return f"{starts[i]:.0f}-{ends[i]:.0f} s"

print(f"longest stretch above the 2 s target: {longest_late_stretch(df.arrived_at.to_numpy(), w)}")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df.arrived_at, w)
ax.axhline(WAIT_TARGET_S, color="k", linestyle="--", linewidth=0.8, label="2 s target")
ax.set_xlabel("trace time (s)")
ax.set_ylabel("predicted wait (s)")
ax.legend(frameon=False)
plt.show()

## KV-cache memory in use (one GPU)

In [ ]:
print(f"KV per token: {kv_bytes_per_token() / 1024:.0f} KiB   model weights: {wl.MODEL_WEIGHTS_GIB} GiB (transformer layers only)")

kv = kv_in_use(df, w, BIN_S)
print(f"KV in use (estimate): mean {kv.kv_gib.mean():.1f} GiB   peak {kv.kv_gib.max():.1f} GiB "
      f"at t={kv.bin_start[kv.kv_gib.idxmax()]:.0f}s")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(kv.bin_start, kv.kv_gib)
ax.set_xlabel("trace time (s)")
ax.set_ylabel("estimated KV cache in use (GiB)")
plt.show()

## Questions

The wait target is **P99 under 2 s**: 99% of requests should wait less than 2 s.

**Q1.** Does your model predict that one GPU meets the 2 s target? If not, when does it miss, and is that stretch driven by more requests, longer prompts, or both? Cite the arrival plot.

**Your answer:**

**Q2.** What are the average and the peak KV-cache memory in use, and when is the peak? Which of these GPUs has enough memory for this trace: 24 GiB (L4), 40 GiB (A100), 48 GiB (L40S), 80 GiB (A100 / H100)? Count the model weights and keep 10% of memory free. Which would you pick if you sized by the average instead? Then rerun `kv_in_use` with `max_batch_size` set very large: what peak do you get? Which of the two peaks better estimates what the GPU actually holds, and why?

**Your answer:**

In [ ]:
# Q2 scratch

**Q3.** At 128 tokens per 30 ms step, one GPU decodes at most 4,267 tokens/s. What is that throughput if a step takes 20 ms or 50 ms, or if the batch holds only 64? Rerun `predict_queue_wait` with each. How much does the P99 wait move compared with the change in throughput, and why? Then set `decode_tok_per_s=float("inf")`: in this model, can faster decode alone meet the 2 s target? If not, what is the bottleneck?

**Your answer:**

In [ ]:
# Q3 scratch: predict_queue_wait(df, decode_tok_per_s=batch / step_s)
# infinitely fast decode: predict_queue_wait(df, decode_tok_per_s=float("inf"))

**Q4.** Record your predictions for Part 2: the P99 wait, the longest stretch above the 2 s target (printed above), and the peak KV-cache memory (GiB).

| Prediction | Value |
|---|---|
| P99 wait | |
| Longest stretch above the 2 s target | |
| Peak KV-cache memory (GiB) | |